# RCATT — T1 Training on Google Colab

**Before running:** Make sure Runtime → Change runtime type → T4 GPU is selected.

**Drive folder expected:**
```
MyDrive/rcatt_sim/data/train.h5
MyDrive/rcatt_sim/data/val.h5
MyDrive/rcatt_sim/data/test.h5
MyDrive/rcatt_sim/data/ood.h5
```

Outputs (checkpoints + CSV) are saved back to `MyDrive/rcatt_sim/out/t1/`.

In [ ]:
# ── 1. Mount Google Drive ────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── 2. Clone / update the repo ───────────────────────────────────────────────
import os

REPO_DIR = '/content/RCATT-sim'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/coledickerson13/RCATT-sim.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())

In [ ]:
# ── 3. Install dependencies ──────────────────────────────────────────────────
!pip install -q h5py scipy

import torch
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))

In [ ]:
# ── 4. Point data at Drive; create output folder ─────────────────────────────
DRIVE_ROOT = '/content/drive/MyDrive/rcatt_sim'

DATA_DIR = os.path.join(DRIVE_ROOT, 'data')
OUT_DIR  = os.path.join(DRIVE_ROOT, 'out', 't1')
os.makedirs(OUT_DIR, exist_ok=True)

# Verify data files exist
for split in ['train', 'val', 'test', 'ood']:
    p = os.path.join(DATA_DIR, f'{split}.h5')
    size_mb = os.path.getsize(p) / 1e6
    print(f'  {split:5s}.h5  {size_mb:7.1f} MB  ✓')

In [ ]:
# ── 5. Run T1 training ───────────────────────────────────────────────────────
# Outputs:  OUT_DIR/best_model.pt   ← best validation checkpoint
#           OUT_DIR/last_model.pt   ← resume checkpoint (every 10 epochs)
#           OUT_DIR/loss_history.csv
#
# To resume after a session timeout:
#   change --resume to point at last_model.pt (already filled in below)

RESUME = ''   # set to f'--resume {OUT_DIR}/last_model.pt' to resume

!python train_t1.py \
    --train {DATA_DIR}/train.h5 \
    --val   {DATA_DIR}/val.h5 \
    --out   {OUT_DIR} \
    {RESUME}

In [ ]:
# ── 6. Plot training curves ──────────────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt

csv_path = os.path.join(OUT_DIR, 'loss_history.csv')
df = pd.read_csv(csv_path)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(df['epoch'], df['train_loss'],  label='train')
axes[0].plot(df['epoch'], df['val_loss'],    label='val')
axes[0].set_title('Total loss');  axes[0].legend();  axes[0].set_xlabel('epoch')

axes[1].plot(df['epoch'], df['train_state'], label='train')
axes[1].plot(df['epoch'], df['val_state'],   label='val')
axes[1].set_title('State loss (L1)');  axes[1].legend();  axes[1].set_xlabel('epoch')

axes[2].plot(df['epoch'], df['train_exist'], label='train')
axes[2].plot(df['epoch'], df['val_exist'],   label='val')
axes[2].set_title('Existence loss (BCE)');  axes[2].legend();  axes[2].set_xlabel('epoch')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 't1_loss_curves.png'), dpi=150)
plt.show()

best_row = df.loc[df['val_loss'].idxmin()]
print(f"Best epoch: {int(best_row['epoch'])}  |  val_loss: {best_row['val_loss']:.5f}  "
      f"|  val_state: {best_row['val_state']:.5f}  |  val_exist: {best_row['val_exist']:.5f}")